# Location-Aware Relative Depth Mapping of a Room Using 9 Horizontally Captured Images

*(Jupyter version — reads images from a local folder instead of Colab's upload widget)*

This notebook builds a **classical computer vision** pipeline (OpenCV + NumPy, no deep learning) that estimates a *relative* depth map of a room from 9 overlapping images captured by a single camera moving approximately horizontally.

**Before running:** create a folder named `images` next to this notebook (or edit `IMAGE_DIR` in Cell 2) and place your 9 room photos inside it. Name the files so that sorting them alphabetically matches the order you physically captured them in (e.g. `room_01.jpg` ... `room_09.jpg`), since that alphabetical order is what gets assigned to I1 -> I9.

**I5 is the reference image.** Every final coordinate `(u, v)` refers to a pixel location in I5, and the final output is a set of `(u, v, Zr)` triples where `Zr` is a normalized, disparity-based relative depth / nearness value in `[0, 1]` — **not** metric depth.

```
Zr(u,v) = (d(u,v) - d_min) / (d_max - d_min)
```

`Zr = 0` -> relatively far, `Zr = 1` -> relatively near.

**Requirements:** `opencv-python`, `numpy`, `matplotlib`, `pandas` installed in the kernel you run this notebook with (e.g. `pip install opencv-python numpy matplotlib pandas`).


## CELL 1 — Imports

In [4]:
%pip install opencv-python
%pip install numpy matplotlib pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.1 MB/s eta 0:00:00m eta 0:00:010:00:01m
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings

# Ensures plots render inline in classic Jupyter Notebook / JupyterLab
%matplotlib inline

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

OpenCV version: 5.0.0
NumPy version: 2.2.6
Pandas version: 2.3.3


**Explanation:** We import only the libraries we actually need: OpenCV for classical computer vision (feature detection, homographies, stereo matching), NumPy for array math, Matplotlib for visualization, and Pandas for the final tabular output. `os` is used to read images from a local folder (Jupyter has no browser upload widget like Colab's `files.upload()`), and `warnings` keeps numeric output clean when fusing multiple views later on.

## CELL 2 — Load 9 images from a local folder

In [ ]:
# Put your 9 room photos in this folder before running this cell (relative to
# this notebook's working directory, or use an absolute path).
IMAGE_DIR = "./images"

if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(
        f"Image directory '{IMAGE_DIR}' does not exist. "
        f"Create it and place your 9 room photos inside, or edit IMAGE_DIR above."
    )

valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
found_files = sorted(
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(valid_extensions)
)

print(f"Looking for images in: {os.path.abspath(IMAGE_DIR)}")
print(f"Found {len(found_files)} image file(s):")
for f in found_files:
    print(" -", f)

if len(found_files) != 9:
    raise ValueError(
        f"Expected exactly 9 images in '{IMAGE_DIR}', but found {len(found_files)}. "
        f"Make sure exactly 9 image files (and nothing else) are in that folder."
    )
else:
    print("\nSuccess: 9 image files found.")

print("\nNOTE: files are sorted alphabetically and assigned to I1 -> I9 in that order "
      "(see the next cell). Name your files so alphabetical order matches the order "
      "you physically captured them in (e.g. room_01.jpg ... room_09.jpg), or edit "
      "filename_mapping manually in the next cell if that is not the case.")

**Explanation:** Jupyter has no built-in browser upload widget, so instead of Colab's `files.upload()` we simply point the notebook at a local folder and verify it contains exactly 9 image files, per the debugging requirement of failing loudly instead of silently continuing with the wrong number of images.

## CELL 3 — Load images

In [ ]:
# Explicit mapping from image label (I1..I9) to filename.
# By default we assume alphabetical filename order matches physical capture order I1 -> I9.
# If that is not the case, edit this dictionary manually before re-running the cell.
image_order = [f"I{i}" for i in range(1, 10)]
filename_mapping = dict(zip(image_order, found_files))

print("Filename mapping (edit filename_mapping above if this is incorrect):")
for k, v in filename_mapping.items():
    print(f"  {k} -> {v}")

images = {}
image_shapes = {}

for label in image_order:
    fname = filename_mapping[label]
    fpath = os.path.join(IMAGE_DIR, fname)
    img_bgr = cv2.imread(fpath, cv2.IMREAD_COLOR)

    if img_bgr is None:
        raise ValueError(f"Failed to load image for {label} ({fpath}). "
                          f"Make sure it is a valid, readable image file.")

    images[label] = img_bgr
    image_shapes[label] = img_bgr.shape
    print(f"{label} ({fname}): shape = {img_bgr.shape}, dtype = {img_bgr.dtype}")

if len(images) != 9:
    raise ValueError("Not all 9 images were loaded successfully.")

# Check dimension consistency; resize to a common resolution if needed
shapes = set((s[0], s[1]) for s in image_shapes.values())
if len(shapes) > 1:
    print("\nWarning: images have different dimensions:", shapes)
    target_h = min(s[0] for s in image_shapes.values())
    target_w = min(s[1] for s in image_shapes.values())
    print(f"Resizing all images to a common resolution: ({target_w} x {target_h})")
    for label in image_order:
        images[label] = cv2.resize(images[label], (target_w, target_h), interpolation=cv2.INTER_AREA)
    print("Resizing complete. A common resolution is required because feature matching, "
          "homography warping, and StereoSGBM all assume consistent pixel grids across images.")
else:
    print("\nAll images share the same dimensions:", shapes.pop())

REFERENCE_LABEL = "I5"
print(f"\nReference image for this project: {REFERENCE_LABEL}")

**Explanation:** We load every image from disk with OpenCV, store them in an ordered dictionary `images["I1"]` ... `images["I9"]`, and check that each one loaded successfully. If image dimensions differ we resize everything to a shared resolution, since every later step (feature matching, homography warping, stereo matching) requires images to live on comparable pixel grids. `REFERENCE_LABEL` marks I5 as the fixed coordinate system for the whole project.

## CELL 4 — Display all images

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for idx, label in enumerate(image_order):
    row, col = divmod(idx, 3)
    img_rgb = cv2.cvtColor(images[label], cv2.COLOR_BGR2RGB)
    axes[row, col].imshow(img_rgb)
    title = label
    if label == REFERENCE_LABEL:
        title += "  <-- REFERENCE IMAGE"
    axes[row, col].set_title(
        title, fontsize=11,
        color="red" if label == REFERENCE_LABEL else "black",
        fontweight="bold" if label == REFERENCE_LABEL else "normal"
    )
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()

**Explanation:** A quick visual sanity check that all 9 images loaded correctly and are in the expected order. I5 is highlighted in red/bold since every later measurement will ultimately be expressed in its coordinate system.

## CELL 5 — Camera movement explanation

In [ ]:
print("Camera movement sequence:")
print(" -> ".join(image_order))
print()
print("Assumptions for this project:")
print(" 1. A single camera photographed the SAME static room.")
print(" 2. The camera moved approximately along a horizontal axis between shots.")
print(" 3. The camera height and orientation stayed approximately constant.")
print(" 4. Consecutive images have substantial overlap of the same scene.")
print(f" 5. {REFERENCE_LABEL} is the REFERENCE IMAGE: every final coordinate (u, v) "
      f"in this project refers to pixel locations in {REFERENCE_LABEL}.")
print(" 6. No moving objects and no lighting changes are assumed between shots.")

**Explanation:** This cell documents (rather than computes) the physical capture assumptions the whole pipeline depends on. These assumptions are not enforced by code — they must be true of how the photos were actually taken — but stating them clearly is part of being honest about this project's limitations.

## CELL 6 — Convert to grayscale

In [ ]:
gray_images = {}
for label in image_order:
    gray_images[label] = cv2.cvtColor(images[label], cv2.COLOR_BGR2GRAY)
    print(f"{label}: grayscale shape = {gray_images[label].shape}")

**Explanation:** ORB feature detection and StereoSGBM both operate on single-channel intensity images, so we convert every image to grayscale once and reuse it everywhere below.

## CELL 7 — Feature matching / image relationship

In [ ]:
neighbor_pairs = [(f"I{i}", f"I{i+1}") for i in range(1, 9)]
print("Neighboring pairs used for feature matching:")
print(neighbor_pairs)

orb = cv2.ORB_create(nfeatures=3000)

def match_features(labelA, labelB, ratio=0.75):
    """Detect ORB features in two grayscale images and match them with Lowe's ratio test."""
    kpA, desA = orb.detectAndCompute(gray_images[labelA], None)
    kpB, desB = orb.detectAndCompute(gray_images[labelB], None)

    if desA is None or desB is None:
        raise RuntimeError(f"No ORB descriptors found for pair ({labelA}, {labelB}). "
                            f"Check that the images have enough texture.")

    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    knn_matches = bf.knnMatch(desA, desB, k=2)

    good_matches = []
    for pair in knn_matches:
        if len(pair) == 2:
            m, n = pair
            if m.distance < ratio * n.distance:
                good_matches.append(m)

    return kpA, kpB, good_matches

feature_matches = {}
for labelA, labelB in neighbor_pairs:
    kpA, kpB, good = match_features(labelA, labelB)
    feature_matches[(labelA, labelB)] = (kpA, kpB, good)
    print(f"{labelA}-{labelB}: {len(good)} good matches (ratio test)")

# Visualize matches for a couple of example pairs
example_pairs = [("I4", "I5"), ("I5", "I6")]
fig, axes = plt.subplots(len(example_pairs), 1, figsize=(14, 5 * len(example_pairs)))
if len(example_pairs) == 1:
    axes = [axes]

for ax, (labelA, labelB) in zip(axes, example_pairs):
    kpA, kpB, good = feature_matches[(labelA, labelB)]
    match_img = cv2.drawMatches(images[labelA], kpA, images[labelB], kpB,
                                 good[:60], None,
                                 flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    ax.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Feature matches: {labelA} <-> {labelB} ({len(good)} matches)")
    ax.axis("off")

plt.tight_layout()
plt.show()

**Explanation:** Because the camera physically moved between shots, the same wall corner or object edge lands on different pixel coordinates in different images. ORB (a fast, classical, patent-free feature detector) finds distinctive keypoints, and a brute-force Hamming-distance matcher with a ratio test links matching keypoints between each neighboring pair. This gives us the raw correspondences needed to later relate every image's coordinates to I5's coordinates.

## CELL 8 — Establish reference relationship to I5

In [ ]:
def estimate_homography(labelA, labelB, min_matches=15):
    """Estimate a homography mapping points from labelA's image plane to labelB's image plane."""
    kpA, kpB, good = feature_matches[(labelA, labelB)]

    if len(good) < min_matches:
        raise RuntimeError(f"Not enough good matches between {labelA} and {labelB} "
                            f"({len(good)} < {min_matches}). Increase overlap or texture.")

    ptsA = np.float32([kpA[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    ptsB = np.float32([kpB[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(ptsA, ptsB, cv2.RANSAC, ransacReprojThreshold=3.0)
    inliers = int(mask.sum()) if mask is not None else 0

    if H is None:
        raise RuntimeError(f"Homography estimation failed for {labelA}->{labelB}.")

    print(f"{labelA} -> {labelB}: {len(good)} matches, {inliers} RANSAC inliers "
          f"({100 * inliers / len(good):.1f}% inlier ratio)")

    return H

# Homography for each neighboring pair, mapping FROM the first label TO the second label
pairwise_homographies = {}
for labelA, labelB in neighbor_pairs:
    pairwise_homographies[(labelA, labelB)] = estimate_homography(labelA, labelB)

def get_pairwise_homography(label_from, label_to):
    """Return the homography mapping label_from's plane to label_to's plane, for adjacent labels."""
    if (label_from, label_to) in pairwise_homographies:
        return pairwise_homographies[(label_from, label_to)]
    elif (label_to, label_from) in pairwise_homographies:
        return np.linalg.inv(pairwise_homographies[(label_to, label_from)])
    else:
        raise KeyError(f"No direct homography available between {label_from} and {label_to}.")

def get_homography_to_I5(label):
    """
    Compose pairwise homographies to map points from `label`'s image plane into
    I5's image plane: p_I5 (homogeneous) ~ H @ p_label (homogeneous).
    """
    if label == REFERENCE_LABEL:
        return np.eye(3)

    idx = int(label[1:])
    ref_idx = int(REFERENCE_LABEL[1:])
    H = np.eye(3)

    if idx < ref_idx:
        for i in range(idx, ref_idx):
            H = pairwise_homographies[(f"I{i}", f"I{i+1}")] @ H
    else:
        for i in range(idx, ref_idx, -1):
            H_step_inv = np.linalg.inv(pairwise_homographies[(f"I{i-1}", f"I{i}")])
            H = H_step_inv @ H

    return H

homographies_to_I5 = {label: get_homography_to_I5(label) for label in image_order}

print("\nComputed a homography chain from every view into the I5 coordinate system.")
for label in image_order:
    tag = " (identity, this IS I5)" if label == REFERENCE_LABEL else ""
    print(f"  H({label} -> I5) ready{tag}")

print("\nLIMITATION: a homography exactly relates two views only when the scene is planar, "
      "or when the camera undergoes pure rotation about its optical center. For a real 3D "
      "room photographed with camera translation, a homography is only an APPROXIMATION. "
      "We use it here as a practical, classical alignment tool for a controlled horizontal "
      "capture, NOT as a claim of exact geometric equivalence between views.")

**Explanation:** This is where we build the bridge between every image and I5. We first estimate a RANSAC-filtered homography for each neighboring pair, then chain those homographies together (e.g. I1->I2->I3->I4->I5) so any image's pixel coordinates can be projected into I5's coordinate system. We are explicit that this is an approximation, not true 3D geometry, since the room is not a flat plane.

## CELL 9 — Define stereo pairs

In [ ]:
# Each stereo pair is (image_to_warp, reference_frame_image). The reference_frame_image
# is always the neighbor that is closer to (or equal to) I5 in the capture chain, so
# depth information naturally accumulates towards the I5 coordinate system.
stereo_pairs = [
    ("I1", "I2"),
    ("I2", "I3"),
    ("I3", "I4"),
    ("I4", "I5"),
    ("I6", "I5"),
    ("I7", "I6"),
    ("I8", "I7"),
    ("I9", "I8"),
]

print("Stereo pairs used for disparity computation (image_to_warp, reference_frame):")
for warp_label, ref_label in stereo_pairs:
    print(f"  {warp_label}  is aligned onto reference frame {ref_label}")

**Explanation:** We reuse the 8 neighboring pairs, but now explicitly assign, for each pair, which image acts as the fixed 'reference frame' (the one closer to I5) and which image gets warped onto it. This ordering is what lets us walk depth information toward I5 step by step in the cells that follow.

## CELL 10 — StereoSGBM setup

In [ ]:
def create_stereo_matcher(min_disp=0, num_disp=16 * 8, block_size=5):
    """
    Create a reusable StereoSGBM matcher.

    minDisparity: minimum possible disparity value (0, since we already warp the
                  moving image onto the reference frame before matching).
    numDisparities: size of the disparity search range, must be a positive multiple of 16.
    blockSize: matched block size (odd number); larger = smoother but less fine detail.
    P1, P2: penalties that control disparity smoothness between neighboring pixels
            (P2 > P1, following OpenCV's recommended formula based on blockSize).
    uniquenessRatio: percentage margin by which the best match must beat the second-best
                      match; filters out ambiguous matches.
    speckleWindowSize / speckleRange: post-processing that removes small, noisy,
                      disconnected disparity blobs.
    disp12MaxDiff: maximum allowed difference in the left-right consistency check.
    """
    matcher = cv2.StereoSGBM_create(
        minDisparity=min_disp,
        numDisparities=num_disp,
        blockSize=block_size,
        P1=8 * 1 * block_size ** 2,
        P2=32 * 1 * block_size ** 2,
        disp12MaxDiff=1,
        uniquenessRatio=10,
        speckleWindowSize=100,
        speckleRange=2,
        mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY,
    )
    return matcher

stereo_matcher = create_stereo_matcher()

print("StereoSGBM matcher created with the following parameters:")
print(f"  minDisparity      = {stereo_matcher.getMinDisparity()}")
print(f"  numDisparities    = {stereo_matcher.getNumDisparities()}")
print(f"  blockSize         = {stereo_matcher.getBlockSize()}")
print(f"  uniquenessRatio   = {stereo_matcher.getUniquenessRatio()}")
print(f"  speckleWindowSize = {stereo_matcher.getSpeckleWindowSize()}")
print(f"  speckleRange      = {stereo_matcher.getSpeckleRange()}")
print(f"  disp12MaxDiff     = {stereo_matcher.getDisp12MaxDiff()}")

**Explanation:** StereoSGBM (Semi-Global Block Matching) is a classical, non-learning-based stereo algorithm built into OpenCV. We create one shared, well-documented configuration so every pair uses the same, explainable parameters. Remember: OpenCV returns disparity in a fixed-point format that must be divided by 16.0 to get real values — we handle that in the next cell.

## CELL 11 — Calculate disparity

In [ ]:
def warp_to_reference(label_to_warp, reference_label):
    """Warp `label_to_warp`'s grayscale image onto `reference_label`'s pixel grid."""
    H = get_pairwise_homography(label_to_warp, reference_label)
    h, w = gray_images[reference_label].shape

    warped = cv2.warpPerspective(gray_images[label_to_warp], H, (w, h))

    # Track which warped pixels actually came from real image content
    # (vs. black borders introduced by the perspective warp).
    ones = np.ones_like(gray_images[label_to_warp], dtype=np.uint8) * 255
    warped_mask = cv2.warpPerspective(ones, H, (w, h), flags=cv2.INTER_NEAREST)
    valid_warp_mask = warped_mask > 0

    return warped, valid_warp_mask

disparity_maps = {}        # (warp_label, ref_label) -> disparity map, in ref_label's pixel grid
disparity_valid_masks = {} # (warp_label, ref_label) -> boolean mask of real (non-border) content

for label_to_warp, reference_label in stereo_pairs:
    warped_gray, valid_warp_mask = warp_to_reference(label_to_warp, reference_label)
    reference_gray = gray_images[reference_label]

    # After warping, both images share reference_label's pixel grid, so we treat them
    # as an approximately-aligned stereo pair and compute disparity between them.
    raw_disp = stereo_matcher.compute(warped_gray, reference_gray)
    disp = raw_disp.astype(np.float32) / 16.0  # StereoSGBM returns fixed-point disparity

    disparity_maps[(label_to_warp, reference_label)] = disp
    disparity_valid_masks[(label_to_warp, reference_label)] = valid_warp_mask

    valid_pct = 100 * valid_warp_mask.sum() / valid_warp_mask.size
    print(f"{label_to_warp} -> {reference_label}: disparity computed "
          f"(warp coverage = {valid_pct:.1f}% of {reference_label}'s frame)")

**Explanation:** For each stereo pair, we first warp the moving image onto the reference image's pixel grid using the homography from Cell 8 (our practical alignment step, since we have no camera calibration for true rectification). We then run StereoSGBM on the warped image and the reference image. Because the homography only perfectly aligns points on the scene's dominant plane, real depth variation shows up as residual disparity — which is exactly the signal we want. The output disparity map is indexed by reference_label's own pixel coordinates.

## CELL 12 — Visualize disparity

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.ravel()

for ax, (label_to_warp, reference_label) in zip(axes, stereo_pairs):
    disp = disparity_maps[(label_to_warp, reference_label)]
    valid_mask = disparity_valid_masks[(label_to_warp, reference_label)]
    disp_display = np.where(valid_mask, disp, np.nan)

    im = ax.imshow(disp_display, cmap="plasma")
    ax.set_title(f"Disparity: {label_to_warp} <-> {reference_label}\n(in {reference_label}'s frame)")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("Interpretation: larger disparity values = relatively NEARER surfaces.")
print("Smaller disparity values = relatively FARTHER surfaces.")

**Explanation:** A quick visual check of the raw disparity output for every pair before we filter or normalize anything. This is useful for spotting obviously broken pairs (e.g. very little valid coverage) early, per the debugging requirement.

## CELL 13 — Valid disparity masks

In [ ]:
def get_valid_disparity_mask(disp, warp_valid_mask, min_valid_disp=0.0):
    """
    Combine several validity checks into one mask:
      - the pixel came from real warped image content (not a black border), and
      - the disparity value is finite (no NaN / inf), and
      - the disparity value is strictly positive (StereoSGBM marks unmatched
        pixels with a negative sentinel value, e.g. -16 before the /16.0 division).
    """
    finite_mask = np.isfinite(disp)
    positive_mask = disp > min_valid_disp
    return warp_valid_mask & finite_mask & positive_mask

final_valid_masks = {}
for label_to_warp, reference_label in stereo_pairs:
    disp = disparity_maps[(label_to_warp, reference_label)]
    warp_mask = disparity_valid_masks[(label_to_warp, reference_label)]

    valid_mask = get_valid_disparity_mask(disp, warp_mask)
    final_valid_masks[(label_to_warp, reference_label)] = valid_mask

    valid_pct = 100 * valid_mask.sum() / valid_mask.size
    print(f"{label_to_warp} <-> {reference_label}: {valid_pct:.1f}% valid disparity pixels")

**Explanation:** Before we normalize anything, we must remove disparity values that don't represent real correspondences: pixels outside the warped image's real content, and pixels StereoSGBM flagged as unmatched (negative) or non-finite. These invalid pixels are excluded from every later calculation instead of silently polluting the results.

## CELL 14 — Determine robust d_min and d_max

In [ ]:
LOWER_PERCENTILE = 2
UPPER_PERCENTILE = 98

all_valid_disparities = []
for label_to_warp, reference_label in stereo_pairs:
    disp = disparity_maps[(label_to_warp, reference_label)]
    mask = final_valid_masks[(label_to_warp, reference_label)]
    all_valid_disparities.append(disp[mask])

all_valid_disparities = np.concatenate(all_valid_disparities)

if all_valid_disparities.size == 0:
    raise RuntimeError("No valid disparity values found across any stereo pair. "
                        "Check image overlap, scene texture, and StereoSGBM parameters.")

d_min = float(np.percentile(all_valid_disparities, LOWER_PERCENTILE))
d_max = float(np.percentile(all_valid_disparities, UPPER_PERCENTILE))

print(f"Total valid disparity samples (all pairs combined): {all_valid_disparities.size}")
print(f"d_min ({LOWER_PERCENTILE}th percentile) = {d_min:.3f}")
print(f"d_max ({UPPER_PERCENTILE}th percentile) = {d_max:.3f}")

if d_max <= d_min:
    raise RuntimeError("d_max <= d_min, cannot normalize. Check disparity computation upstream.")

**Explanation:** d_min and d_max define the normalization range for the whole room, so they must be robust. We pool valid disparities from every pair and take the 2nd/98th percentile instead of the raw min/max, because StereoSGBM can produce isolated noisy outliers (a handful of badly mismatched pixels) that would otherwise distort the entire normalization.

## CELL 15 — Calculate relative depth

In [ ]:
relative_depth_maps = {}

for label_to_warp, reference_label in stereo_pairs:
    disp = disparity_maps[(label_to_warp, reference_label)]
    mask = final_valid_masks[(label_to_warp, reference_label)]

    zr = np.full(disp.shape, np.nan, dtype=np.float32)
    zr_valid = (disp[mask] - d_min) / (d_max - d_min)
    zr_valid = np.clip(zr_valid, 0.0, 1.0)
    zr[mask] = zr_valid

    relative_depth_maps[(label_to_warp, reference_label)] = zr

    print(f"{label_to_warp} <-> {reference_label}: relative depth computed, "
          f"range = [{np.nanmin(zr):.3f}, {np.nanmax(zr):.3f}]")

print("\nFormula used (fixed, per project definition):")
print("  Zr(u,v) = (d(u,v) - d_min) / (d_max - d_min), clipped to [0, 1]")
print("  Zr = 0 -> relatively far,   Zr = 1 -> relatively near")

**Explanation:** This applies the project's exact, fixed formula — Zr = (d - d_min) / (d_max - d_min) — to every valid disparity pixel, clipped to [0,1]. Invalid pixels stay NaN rather than being assigned a fabricated depth value.

## CELL 16 — Display relative depth maps

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.ravel()

for ax, (label_to_warp, reference_label) in zip(axes, stereo_pairs):
    zr = relative_depth_maps[(label_to_warp, reference_label)]
    im = ax.imshow(zr, cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"Relative depth Zr: {label_to_warp} <-> {reference_label}\n(in {reference_label}'s frame)")
    ax.axis("off")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Zr (0=far, 1=near)")

plt.tight_layout()
plt.show()

**Explanation:** A per-pair visual check of the normalized relative depth before we attempt to merge anything into the I5 coordinate system. The colorbar consistently means 0 = relatively far and 1 = relatively near across every pair.

## CELL 17 — Bring depth information into I5 coordinates

In [ ]:
def warp_depth_map_to_I5(zr_map, source_frame_label):
    """
    Warp a Zr map (currently expressed in `source_frame_label`'s pixel grid) into
    I5's pixel grid, using the homography chain built in Cell 8.

    Nearest-neighbor interpolation is used deliberately: blending Zr values from
    neighboring pixels during warping would mix depth values from potentially
    different physical surfaces, which we explicitly want to avoid.
    """
    if source_frame_label == REFERENCE_LABEL:
        return zr_map.copy()

    H = homographies_to_I5[source_frame_label]
    h, w = gray_images[REFERENCE_LABEL].shape

    # warpPerspective can't handle NaN directly, so we substitute a sentinel value,
    # warp it, then convert the sentinel back to NaN afterwards.
    sentinel = -1.0
    zr_filled = np.where(np.isnan(zr_map), sentinel, zr_map).astype(np.float32)

    warped = cv2.warpPerspective(zr_filled, H, (w, h),
                                  flags=cv2.INTER_NEAREST, borderValue=sentinel)
    warped[warped == sentinel] = np.nan
    return warped

zr_maps_in_I5_frame = {}

for label_to_warp, reference_label in stereo_pairs:
    zr_map = relative_depth_maps[(label_to_warp, reference_label)]
    zr_in_I5 = warp_depth_map_to_I5(zr_map, reference_label)
    zr_maps_in_I5_frame[(label_to_warp, reference_label)] = zr_in_I5

    valid_pct = 100 * np.sum(~np.isnan(zr_in_I5)) / zr_in_I5.size
    print(f"{label_to_warp} <-> {reference_label}: warped from {reference_label}'s frame "
          f"into I5's frame ({valid_pct:.1f}% valid coverage)")

print("\nWhat is being transformed: each pair's Zr map currently lives in its own "
      "reference_label pixel grid. We reuse the homography chain from Cell 8 (built from "
      "real ORB correspondences, refined with RANSAC) to project each Zr value to the I5 "
      "pixel location it geometrically corresponds to. Nearest-neighbor sampling avoids "
      "blending two different surfaces' Zr values together. Regions with no valid "
      "correspondence remain NaN rather than being invented.")

**Explanation:** This is the critical step that satisfies the project's core requirement: every measurement, regardless of which pair produced it, is projected into I5's pixel coordinate system before we do anything else with it. We are not simply resizing maps or assuming shared coordinates — we use the actual estimated geometry.

## CELL 18 — Fuse the multi-view information

In [ ]:
stacked_zr = np.stack(list(zr_maps_in_I5_frame.values()), axis=0)  # shape: (8, H, W)
valid_counts = np.sum(~np.isnan(stacked_zr), axis=0)

def fuse_observations(stacked, outlier_std_threshold=2.0):
    """
    Fuse multiple Zr observations per I5 pixel:
      1. Compute the per-pixel mean and std across all views (ignoring NaN).
      2. Reject observations farther than `outlier_std_threshold` standard
         deviations from the per-pixel mean (obvious outliers).
      3. Take the median of the remaining inlier observations.
    Pixels with zero valid observations stay NaN — we never fabricate depth.
    """
    mean_per_pixel = np.nanmean(stacked, axis=0)
    std_per_pixel = np.nanstd(stacked, axis=0)

    deviation = np.abs(stacked - mean_per_pixel[np.newaxis, :, :])
    is_outlier = deviation > (outlier_std_threshold * std_per_pixel[np.newaxis, :, :] + 1e-6)

    cleaned = np.where(is_outlier, np.nan, stacked)
    fused = np.nanmedian(cleaned, axis=0)
    return fused.astype(np.float32)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    final_depth_I5 = fuse_observations(stacked_zr)

n_valid_final = int(np.sum(~np.isnan(final_depth_I5)))
n_any_observation = int(np.sum(valid_counts > 0))

print("Fusion complete.")
print(f"I5 pixels with at least one valid observation: {n_any_observation} "
      f"({100 * n_any_observation / valid_counts.size:.1f}%)")
print(f"I5 pixels with a final fused value: {n_valid_final} "
      f"({100 * n_valid_final / final_depth_I5.size:.1f}%)")
if n_any_observation > 0:
    print(f"Average overlapping observations per pixel (where any exist): "
          f"{valid_counts[valid_counts > 0].mean():.2f}")

**Explanation:** Different stereo pairs contribute overlapping-but-different observations of the same I5 pixel locations. Rather than blindly averaging every map, we reject obvious outliers per pixel (values far from that pixel's own mean across views) and take the median of the remaining values — a simple, explainable, robust fusion strategy that only combines observations that are genuinely mapped to the same I5 location.

## CELL 19 — Final location-aware data

In [ ]:
valid_rows_mask = ~np.isnan(final_depth_I5)
v_coords, u_coords = np.where(valid_rows_mask)
zr_values = final_depth_I5[valid_rows_mask]

location_aware_df = pd.DataFrame({
    "u": u_coords,
    "v": v_coords,
    "relative_depth": zr_values
})
location_aware_df = location_aware_df.sort_values(["v", "u"]).reset_index(drop=True)

print(f"Final location-aware DataFrame: {len(location_aware_df)} rows")
print("NOTE: u and v are pixel coordinates in I5, the reference image.")
location_aware_df.head(10)

**Explanation:** The final data product of this project: a table of (u, v, relative_depth) where u and v are always I5 pixel coordinates, exactly as required. This DataFrame is what gets queried, visualized, and saved in the remaining cells.

## CELL 20 — Query a location

In [ ]:
def query_location(u, v):
    h, w = final_depth_I5.shape
    if not (0 <= u < w and 0 <= v < h):
        print(f"Pixel ({u},{v}) is outside I5's bounds (width={w}, height={h}).")
        return None

    zr = final_depth_I5[v, u]

    print(f"Reference image: {REFERENCE_LABEL}")
    print(f"Pixel: (u={u}, v={v})")

    if np.isnan(zr):
        print("No valid depth information available at this location "
              "(insufficient stereo overlap / correspondence).")
        return None

    print(f"Relative depth Zr: {zr:.4f}")
    if zr > 0.66:
        interpretation = "relatively NEAR"
    elif zr > 0.33:
        interpretation = "MIDDLE distance"
    else:
        interpretation = "relatively FAR"
    print(f"Interpretation: {interpretation}")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(cv2.cvtColor(images[REFERENCE_LABEL], cv2.COLOR_BGR2RGB))
    ax.scatter([u], [v], c="red", s=120, marker="x", linewidths=3)
    ax.set_title(f"Queried location on {REFERENCE_LABEL}: (u={u}, v={v}), Zr={zr:.3f}")
    ax.axis("off")
    plt.show()

    return zr

# Example query -- edit query_u / query_v and re-run this cell to check a different pixel
query_u = final_depth_I5.shape[1] // 2
query_v = final_depth_I5.shape[0] // 2
query_location(query_u, query_v)

**Explanation:** A small interactive-style utility: give it a pixel (u, v) in I5 and it reports the fused relative depth at that location, an interpretation (near/middle/far), and marks the point on I5 so you can visually confirm it makes sense.

## CELL 21 — Final room relative-depth map

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

ax[0].imshow(cv2.cvtColor(images[REFERENCE_LABEL], cv2.COLOR_BGR2RGB))
ax[0].set_title(f"{REFERENCE_LABEL} (Reference Image)")
ax[0].axis("off")

im = ax[1].imshow(final_depth_I5, cmap="viridis", vmin=0, vmax=1)
ax[1].set_title(f"Final Location-Aware Relative Depth (aligned to {REFERENCE_LABEL})")
ax[1].axis("off")
cbar = plt.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)
cbar.set_label("Zr  (0 = relatively far, 1 = relatively near)")

plt.tight_layout()
plt.show()

**Explanation:** The headline visual result of the whole project: I5 side-by-side with the fused, multi-view relative depth map aligned to it, using the same 0=far / 1=near colorbar convention used throughout.

## CELL 22 — Location-aware 3D visualization

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3D projection)

MAX_POINTS_FOR_3D = 15000

df_for_plot = location_aware_df
if len(df_for_plot) > MAX_POINTS_FOR_3D:
    df_for_plot = df_for_plot.sample(MAX_POINTS_FOR_3D, random_state=42)
    print(f"Subsampled to {MAX_POINTS_FOR_3D} points for 3D visualization "
          f"(out of {len(location_aware_df)} total).")

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    df_for_plot["u"], df_for_plot["v"], df_for_plot["relative_depth"],
    c=df_for_plot["relative_depth"], cmap="viridis", s=2, vmin=0, vmax=1
)

ax.set_xlabel("Horizontal pixel location (u)")
ax.set_ylabel("Vertical pixel location (v)")
ax.set_zlabel("Relative depth (Zr)")
ax.set_title("Location-aware relative depth of the room (I5 reference)\n"
             "NOTE: this is NOT physical 3D space -- axes are pixel coords + normalized Zr")
ax.invert_yaxis()  # image v grows downward; inverting gives a more intuitive view

fig.colorbar(scatter, ax=ax, shrink=0.6, label="Zr")
plt.tight_layout()
plt.show()

**Explanation:** A 3D scatter plot of every (u, v, Zr) point. The axes are explicitly labeled as pixel coordinates plus normalized relative depth -- not physical X/Y/Z -- to avoid any impression that this is a metric 3D reconstruction of the room.

## CELL 23 — Save results

In [ ]:
output_image_path = "final_relative_depth_I5.png"
output_csv_path = "location_aware_depth_I5.csv"

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(final_depth_I5, cmap="viridis", vmin=0, vmax=1)
ax.set_title(f"Final Relative Depth Map (aligned to {REFERENCE_LABEL})")
ax.axis("off")
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Zr (0=far, 1=near)")
plt.tight_layout()
plt.savefig(output_image_path, dpi=150)
plt.show()
print(f"Saved final depth map image to: {output_image_path}")

location_aware_df.to_csv(output_csv_path, index=False)
print(f"Saved location-aware CSV to: {output_csv_path}")
print(f"CSV columns: {list(location_aware_df.columns)}")

# Optional: save raw per-pair disparity maps for later inspection
raw_disp_stack = np.array([disparity_maps[p] for p in stereo_pairs], dtype=object)
np.save("disparity_maps_raw.npy", raw_disp_stack, allow_pickle=True)
print("Saved raw per-pair disparity maps to: disparity_maps_raw.npy")

print(f"\nAll files were saved to the notebook's working directory: {os.getcwd()}")

**Explanation:** We save the final visual depth map and the (u, v, relative_depth) CSV to disk, plus the raw per-pair disparity maps for anyone who wants to inspect the intermediate results. Unlike Colab, plain Jupyter has no browser-download step — the files are simply written to the notebook's working directory, from where you can open or move them normally.

## CELL 24 — Final explanation

In [ ]:
summary = """
COMPLETE PROJECT PIPELINE SUMMARY
==================================

9 images (I1..I9)
        |
Horizontal camera movement (same room, same camera, same approx. height/orientation)
        |
Neighboring stereo pairs (I1-I2, I2-I3, ..., I8-I9)
        |
ORB feature matching + RANSAC-filtered homography estimation
        |
StereoSGBM disparity computation, d(u,v), per pair (after homography-based warping)
        |
Valid disparity filtering (remove negative / zero / NaN / border pixels)
        |
Robust d_min / d_max via percentiles
        |
Normalized relative depth:  Zr = (d - d_min) / (d_max - d_min), clipped to [0,1]
        |
Homography-chain warp of every pair's Zr map into I5's coordinate system
        |
Multi-view fusion (outlier rejection + median) at each I5 pixel
        |
FINAL LOCATION-AWARE ROOM MAP:  (u, v, Zr)   -- u, v always in I5's pixel grid
"""
print(summary)

print("IMPORTANT LIMITATIONS OF THIS PROJECT:")
print("- This is a RELATIVE depth project. Zr values are NOT depth in metres.")
print("- Zr=0.5 means the disparity is halfway between d_min and d_max in disparity")
print("  space -- it does NOT mean the surface is physically halfway between the")
print("  nearest and farthest points in the room.")
print("- Homographies are an approximation for a real 3D room (exact only for a")
print("  planar scene or pure camera rotation). They were used as a practical,")
print("  classical alignment tool, NOT as true stereo rectification.")
print("- Accuracy depends on: camera motion accuracy, image overlap, scene texture,")
print("  lighting consistency, image quality, StereoSGBM parameters, and the absence")
print("  of moving objects in the room.")
print("- Textureless surfaces (e.g. blank walls) typically produce sparse or")
print("  unreliable disparity, and therefore sparse Zr coverage at those locations.")
print("- No deep learning, no metric reconstruction, no SLAM, and no pretrained")
print("  depth model were used anywhere in this pipeline -- classical CV only.")

**Explanation:** A single readable summary of the entire pipeline from raw images to the final (u, v, Zr) room map, plus a restatement of the project's honest limitations, so anyone reading the notebook understands exactly what was built and what it does and does not represent.